In [1]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal

# =========================
# LOAD DATASETS
# =========================

trades = pd.read_csv("historical_data.csv")
sentiment = pd.read_csv("fear_greed_index.csv")

print("Trades Shape:", trades.shape)
print("Sentiment Shape:", sentiment.shape)

# =========================
# DATA CLEANING
# =========================

trades['Timestamp IST'] = pd.to_datetime(
    trades['Timestamp IST'],
    dayfirst=True,
    errors='coerce'
)

sentiment['date'] = pd.to_datetime(
    sentiment['date'],
    errors='coerce'
)

trades['date'] = trades['Timestamp IST'].dt.date
sentiment['date'] = sentiment['date'].dt.date

# =========================
# MERGE DATASETS
# =========================

merged = pd.merge(
    trades,
    sentiment[['date', 'classification']],
    on='date',
    how='left'
)

print("\nMerged Shape:", merged.shape)

# =========================
# BASIC INFO
# =========================

print("\nTotal Trades:", len(merged))
print("Unique Traders:", merged['Account'].nunique())
print("Unique Coins:", merged['Coin'].nunique())

# =========================
# PNL ANALYSIS
# =========================

print("\nProfitability by Sentiment")

pnl_summary = merged.groupby(
    'classification'
)['Closed PnL'].agg([
    'count',
    'mean',
    'median',
    'sum'
]).round(2)

print(pnl_summary)

# =========================
# WIN RATE
# =========================

merged['Win'] = merged['Closed PnL'] > 0

win_rate = (
    merged.groupby('classification')['Win']
    .mean()
    .mul(100)
    .round(2)
)

print("\nWin Rate (%)")
print(win_rate)

# =========================
# VOLUME ANALYSIS
# =========================

volume = (
    merged.groupby('classification')['Size USD']
    .sum()
    .round(2)
)

print("\nTrading Volume")
print(volume)

# =========================
# BUY VS SELL
# =========================

buy_sell = pd.crosstab(
    merged['classification'],
    merged['Side']
)

print("\nBuy vs Sell")
print(buy_sell)

# =========================
# ROI ANALYSIS
# =========================

merged['ROI'] = (
    merged['Closed PnL'] /
    merged['Size USD']
)

roi = (
    merged.groupby('classification')['ROI']
    .mean()
    .round(6)
)

print("\nROI Analysis")
print(roi)

# =========================
# TOP TRADERS
# =========================

top_traders = (
    merged.groupby('Account')['Closed PnL']
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 10 Traders")
print(top_traders.head(10))

# =========================
# TOP COINS
# =========================

top_coins = (
    merged.groupby('Coin')['Closed PnL']
    .sum()
    .sort_values(ascending=False)
)

print("\nTop 10 Coins")
print(top_coins.head(10))

# =========================
# KRUSKAL TEST
# =========================

groups = [
    group['Closed PnL'].dropna()
    for _, group in merged.groupby('classification')
]

stat, p_value = kruskal(*groups)

print("\nKruskal-Wallis Test")
print("Statistic:", stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Significant difference between sentiment groups")
else:
    print("No significant difference detected")

# =========================
# EXPORT SUMMARY
# =========================

summary = merged.groupby('classification').agg(
    Trades=('Closed PnL', 'count'),
    Avg_PnL=('Closed PnL', 'mean'),
    Total_PnL=('Closed PnL', 'sum'),
    Avg_Volume=('Size USD', 'mean')
)

summary.to_csv(
    "sentiment_analysis_summary.csv"
)

print("\nSummary exported successfully.")

Trades Shape: (211224, 16)
Sentiment Shape: (2644, 4)

Merged Shape: (211224, 18)

Total Trades: 211224
Unique Traders: 32
Unique Coins: 246

Profitability by Sentiment
                count   mean  median         sum
classification                                  
Extreme Fear    21400  34.54     0.0   739110.25
Extreme Greed   39992  67.89     0.0  2715171.31
Fear            61837  54.29     0.0  3357155.44
Greed           50303  42.74     0.0  2150129.27
Neutral         37686  34.31     0.0  1292920.68

Win Rate (%)
classification
Extreme Fear     37.06
Extreme Greed    46.49
Fear             42.08
Greed            38.48
Neutral          39.70
Name: Win, dtype: float64

Trading Volume
classification
Extreme Fear     1.144843e+08
Extreme Greed    1.244652e+08
Fear             4.833248e+08
Greed            2.885825e+08
Neutral          1.802421e+08
Name: Size USD, dtype: float64

Buy vs Sell
Side              BUY   SELL
classification              
Extreme Fear    10935  10465
Extrem